<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/dentistry/lecture_10/%D0%9F%D1%80%D0%B0%D0%BA%D1%82%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B0%D1%8F_%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%B0_%E2%84%96_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Практическая работа № 10: Агенты и мультиагентные системы в стоматологии

## Введение

В лекции №10 мы познакомились с агентными и мультиагентными системами, которые представляют собой следующий уровень развития ИИ-инструментов для клинической стоматологии. В отличие от простых чат-ботов, агенты способны:

- **планировать** последовательность диагностических и лечебных действий,
- **использовать инструменты** (поиск по МКБ-10, оценка риска одонтогенной инфекции, проверка лекарственных взаимодействий, обращение к базам знаний),
- **запоминать** историю диалога (краткосрочная и долгосрочная память о пациенте),
- **адаптироваться** к жалобам и клинической картине,
- **координировать работу** нескольких специализированных агентов в мультиагентных системах (консилиум).

Мы разобрали, как **LangChain** помогает создавать агентов с памятью и инструментами, а **LangGraph** позволяет строить мультиагентные системы с состоянием, где агенты обмениваются информацией и совместно решают сложные диагностические задачи. Также мы обсудили долгосрочных персонализированных помощников, тренды 2026 года и этические аспекты применения агентов в стоматологической практике.

**Цель данной работы** — закрепить практические навыки:

- установки и настройки **Ollama** в Google Colab,
- создания **агентов** с использованием **LangChain** (инструменты, память),
- построения **мультиагентных систем** с **LangGraph**,
- интеграции **RAG** (векторный поиск с Chroma) в агентные системы,
- сравнения single‑agent и multi‑agent подходов,
- критической оценки этических рисков и разработки протоколов безопасного использования в стоматологии.

---

## Подготовка рабочей среды

Все задания выполняются в **Google Colab** (бесплатная облачная среда с GPU). Вам потребуется аккаунт Google и доступ к интернету.

**Минимальные требования:**

- Браузер (Chrome, Firefox, Edge).
- Аккаунт Google для доступа к Colab.
- Около 5–7 ГБ свободного места в Google Drive (для сохранения моделей, опционально).

**Важно:** все примеры кода адаптированы для Colab и используют модель `llama3.1:8b` (около 4.7 ГБ) или `llama3.2:3b` (более лёгкая). Если у вас медленный интернет, можно заменить на `llama3.2:1b`.

Перед началом работы создайте новый ноутбук в Colab и переименуйте его в `Agents_Dentistry`.

---

## Часть 1. Теоретические вопросы (для самопроверки)

Письменно ответьте на следующие вопросы (кратко, но содержательно). Это поможет убедиться, что вы понимаете ключевые концепции.

1. Что такое агент в контексте LLM? В чём его отличие от обычного чат-бота? Приведите аналогию из стоматологической практики (например, ассистент врача).

2. Назовите четыре основных компонента агента. Как каждый из них соотносится с работой врача-стоматолога (например, инструменты = рентген, ЭОД, пародонтальный зонд)?

3. Что такое краткосрочная и долгосрочная память агента? Приведите пример использования в стоматологическом лечении (например, история лечения зуба, аллергии, принимаемые препараты).

4. Как LangChain упрощает создание агентов? Какие компоненты LangChain используются для инструментов, памяти и планирования?

5. Что такое LangGraph и для чего он нужен? Чем он отличается от простой цепочки (chain) в LangChain? Приведите пример маршрутизации пациента в многопрофильной клинике.

6. Опишите архитектуру мультиагентной системы с четырьмя агентами: стоматолог-интервьюер, диагност, рентгенолог (или библиотекарь RAG), супервизор. Как они кооперируются?

7. Что такое Human‑in‑the‑Loop (HITL) и почему он критически важен в клинических агентных системах (например, при подозрении на флегмону или кровотечение)?

8. Назовите три тренда 2026 года в области долгосрочных персонализированных ИИ-помощников для стоматологии (например, мониторинг заживления, напоминания о гигиене).

9. Какие этические риски связаны с использованием агентов, которые самостоятельно вызывают инструменты (например, оценку риска одонтогенной инфекции или проверку лекарственных взаимодействий)?

10. **Рефлексивный вопрос:** как вы видите баланс между автономностью агента и человеческим контролем в диагностике стоматологических заболеваний? Где должна проходить граница?

---

## Часть 2. Практические задания на Python

Все задания выполняйте в одном Jupyter Notebook в Colab. Код должен быть снабжён комментариями на русском языке. В конце каждого задания приводите краткий анализ результатов.


### Задание 1. Установка и настройка Ollama в Colab

**Описание.** В этом задании вы подготовите окружение: установите Ollama, запустите сервер и скачаете модель.

**Требуется:**

1. Выполните код из лекции (раздел 7.2 «Подготовка окружения»), который:
   - удаляет старые файлы Ollama,
   - скачивает и устанавливает Ollama с GitHub,
   - запускает сервер с проверкой доступности,
   - скачивает модель `llama3.1:8b` (или `llama3.2:3b`).

2. Проверьте, что модель загружена, выполнив `ollama list`.

3. Отправьте тестовый запрос через `ollama run <model> "Привет"` и получите ответ.

**Что сдать:** скриншоты выполнения всех шагов (терминал в Colab) и краткое описание (1 страница).


# Ваш код (вставьте сюда код из лекции, адаптированный под задание):


### Задание 2. Создание агента-интервьюера с одним инструментом (поиск по МКБ-10)

**Описание.** В этом задании вы создадите простого агента с одним инструментом — поиском стоматологических диагнозов по МКБ-10. Агент будет задавать уточняющие вопросы и при необходимости вызывать инструмент.

**Требуется:**

1. Загрузите LLM через `ChatOllama` (используйте модель, скачанную в Задании 1).

2. Создайте инструмент `search_icd10(query)` (как в лекции), который возвращает диагностические критерии для распространённых стоматологических нозологий (например, пульпит, периодонтит, кариес).

3. Настройте память (`ConversationBufferMemory`).

4. Создайте агента с помощью `initialize_agent` с типом `AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION`.

5. Проведите диалог: задайте два вопроса от имени пациента (например, «Болит зуб при накусывании, десна припухла» и «Какие бывают болезни пародонта?»). Выведите ход рассуждений агента (`verbose=True`).

6. Проанализируйте: вызвал ли агент инструмент, когда это было нужно?


# Ваш код решения задачи:


### Задание 3. Добавление памяти и второго инструмента (оценка риска)

**Описание.** Расширьте агента из Задания 2: добавьте инструмент для оценки риска одонтогенной инфекции и настройте память, чтобы агент запоминал предыдущие ответы пациента.

**Требуется:**

1. Создайте второй инструмент `assess_infection_risk(text)`, который по ключевым словам (отёк, тризм, температура, дисфагия) определяет уровень риска (высокий/умеренный/низкий).

2. Добавьте его в список инструментов агента.

3. Модифицируйте системный промпт, чтобы агент при подозрении на инфекцию обязательно вызывал `assess_infection_risk`.

4. Проведите диалог из трёх сообщений:
   - Пациент: «У меня болит зуб и немного припухла десна».
   - Агент: задаёт уточняющий вопрос.
   - Пациент: «Щека тоже начала опухать, и больно глотать».
   - (Агент должен вызвать `assess_infection_risk` и отреагировать.)

5. Выведите историю диалога из памяти агента.

**Что сдать:** код, вывод агента, анализ того, как память повлияла на диалог.


# Ваш код решения задачи:


### Задание 4. Агент-аналитик рентгенологических описаний (или микробиологических отчётов)

**Описание.** Создайте агента, который анализирует текстовые описания рентгеновских снимков (ОПТГ, КЛКТ) или, для ординаторов-микробиологов, результаты бактериологических посевов. Выявляет ключевые паттерны, оценивает состояние и предлагает дополнительные методы. Используйте код из лекции (раздел 7.4) как основу, адаптировав под выбранную специальность.

**Требуется (для стоматологов/рентгенологов):**

1. Определите три инструмента:
   - `detect_radiographic_patterns` — по ключевым словам определяет паттерны (периапикальное просветление, резорбция, остеопороз и т.п.).
   - `suggest_additional_imaging` — рекомендует дополнительные методы (прицельный снимок, КЛКТ, МРТ ВНЧС).
   - `assess_bone_density` — оценивает плотность костной ткани по описанию.

2. Создайте агента с памятью (можно без памяти, так как описания независимы).

3. Протестируйте на трёх описаниях:
   - «На ОПТГ в области 46 зуба периапикальное просветление с нечеткими контурами, резорбция костной ткани в фуркации.»
   - «Прицельный снимок 36: очаг разрежения у апекса, но информации недостаточно для оценки каналов.»
   - «Диффузное снижение плотности костной ткани челюстей, признаки остеопороза.»

4. Для каждого описания выведите анализ агента.

5. Сравните ответы: как агент различает паттерны и плотность?

**Альтернатива для микробиологов:** замените инструменты на `interpret_culture_report`, `assess_antibiotic_resistance`, `recommend_antibiotic`.

**Что сдать:** код, вывод агента, анализ результатов.


# Ваш код решения задачи:


### Задание 5. Мультиагентная система с LangGraph (3 агента: интервьюер, диагност, супервизор)

**Описание.** Постройте мультиагентную систему для диагностики стоматологического заболевания, используя LangGraph. Реализуйте трёх агентов: **Стоматолог-интервьюер**, **Диагност**, **Супервизор** (без библиотекаря с RAG — это будет в Задании 6).

**Требуется:**

1. Определите состояние `DentalAssessmentState` с полями: `patient_text`, `findings` (список), `risk_level`, `recommendation`.

2. Реализуйте агента-интервьюера: извлекает симптомы из текста пациента (по ключевым словам: боль, отёк, кровоточивость, реакция на раздражители) и добавляет их в `findings`.

3. Агента-диагноста: на основе `findings` делает предварительный вывод (например, «соответствует острому пульпиту» или «требуется уточнение»).

4. Агента-супервизора: проверяет риски (по наличию признаков распространённой инфекции, кровотечения) и выдаёт итоговую рекомендацию.

5. Постройте граф: Интервьюер → Диагност → Супервизор.

6. Протестируйте на двух текстах:
   - «Болит зуб внизу слева, больно накусывать, десна припухла, ночью боль усилилась.»
   - «Кровоточат дёсны при чистке, но болей нет, общее состояние нормальное.»

**Что сдать:** код, вывод каждого агента и итоговый отчёт.


# Ваш код решения задачи:


### Задание 6. Интеграция настоящего RAG в мультиагентную систему (агент-библиотекарь)

**Описание.** Добавьте в мультиагентную систему агента-библиотекаря, который использует **настоящий RAG** с векторной базой Chroma и эмбеддингами. Используйте код из лекции (раздел 7.6) для создания базы знаний и функции поиска.

**Требуется:**

1. Создайте векторную базу знаний из 5–7 стоматологических текстов (например, критерии диагнозов по МКБ-10, клинические рекомендации по лечению, протоколы антибиотикопрофилактики).

2. Реализуйте функцию `search_knowledge(query, top_k=2)`, которая возвращает релевантные фрагменты с источниками.

3. Добавьте агента-библиотекаря в граф между диагностом и супервизором. Библиотекарь должен:
   - Использовать `search_knowledge` на основе текста пациента.
   - Добавлять найденные фрагменты в `findings`.

4. Обновите супервизора, чтобы он учитывал информацию из RAG.

5. Протестируйте систему на вопросе, который требует обращения к базе (например, «Каковы критерии диагностики хронического периодонтита?» или «Какие антибиотики рекомендованы при одонтогенной инфекции?»).

**Что сдать:** код всей системы с RAG, пример запроса и ответа, сравнение с системой без RAG.


# Ваш код решения задачи:


### Задание 7. Сравнительный анализ Single Agent vs Multi-Agent System

**Описание.** На основе выполненной работы напишите аналитический отчёт (2–3 страницы), в котором сравните:

- **Single Agent** (Задания 2–4) и **Multi-Agent System** (Задания 5–6).
- Какие задачи каждый подход решает лучше?
- В чём преимущества мультиагентной системы с RAG?
- Какие сложности возникли при реализации?
- Какой подход вы бы рекомендовали для использования в клинической стоматологии и почему? Учтите особенности вашей специальности (терапия, рентгенология, микробиология, неонатология).

**Что сдать:** отчёт в текстовой ячейке Notebook или отдельным файлом.


```
# Ваш отчёт (текст):
```



## Часть 3. Этический анализ

**Задание 8. Разработка этического протокола для агентных систем в стоматологии**

**Описание.** Представьте, что вы — руководитель стоматологической клиники (или отделения лучевой диагностики / микробиологической лаборатории). Вы хотите внедрить мультиагентную систему с RAG для поддержки диагностики и первичного скрининга. Напишите этический протокол (2–3 страницы), в котором осветите:

1. **Информированное согласие** — что пациент должен знать об агентах? Какие пункты обязательно включить (автономность агентов, вызов инструментов, сбор и хранение данных, ограничения ИИ)?

2. **Human‑in‑the‑Loop** — как организовать контроль? В каких случаях агент должен передавать управление человеку (например, признаки флегмоны, подозрение на онкопатологию, беременность, детский возраст)? Кто несёт ответственность за финальный диагноз и назначения?

3. **Конфиденциальность и безопасность** — где хранятся данные? Как обеспечивается анонимизация? Как логируются действия агентов (аудит)? Как защищены рентгеновские снимки и результаты анализов?

4. **Обработка ошибок** — что делать при ложных срабатываниях (например, агент ошибочно оценил риск инфекции как высокий или пропустил взаимодействие лекарств)? Как минимизировать галлюцинации (например, обязательная сверка с клиническими руководствами через RAG)?

5. **Прозрачность** — как объяснить пациенту и коллегам, почему агент принял то или иное решение? Какие записи должны вестись для аудита и разбора клинических случаев?

6. **Обучение персонала** — как подготовить врачей-стоматологов, рентгенологов, микробиологов, неонатологов к работе с агентной системой? Какие навыки нужны (оценка достоверности ИИ, интерпретация результатов, знание ограничений)?

**Что сдать:** этический протокол в формате PDF или текстовой ячейкой в Notebook.


```
# Ваш отчёт (текст):
```


## Часть 4. Дополнительное задание (повышенной сложности — по желанию)

**Задание 9. Создание Streamlit‑приложения для агента-интервьюера**

**Описание.** Разработайте простое веб-приложение на Streamlit, которое позволяет пользователю (пациенту или врачу) общаться с агентом-интервьюером (из Задания 3) через веб-интерфейс.

**Требуется:**

1. Установите Streamlit: `!pip install streamlit` (в Colab можно использовать `streamlit run` через `ngrok` или `colab-ssh`).

2. Создайте приложение с:
   - Полем ввода текста (сообщение пациента).
   - Кнопкой «Отправить».
   - Отображением истории диалога.
   - Индикацией, какой инструмент был вызван (например, «Поиск по МКБ-10», «Оценка риска инфекции»).

3. Запустите приложение и продемонстрируйте его работу (скриншоты).

**Что сдать:** код `app.py`, скриншоты работы.


# Ваш код для Streamlit-приложения:


## Критерии оценки

| Компонент | Процент | Описание |
|-----------|---------|----------|
| Теоретические вопросы (Часть 1) | 15% | Полнота и правильность ответов |
| Задание 1 (установка Ollama) | 5% | Корректность установки, скриншоты |
| Задание 2 (агент с одним инструментом) | 10% | Работающий агент, вызов инструмента |
| Задание 3 (память и второй инструмент) | 10% | Работа памяти, вызов оценки риска |
| Задание 4 (агент-аналитик) | 10% | Корректный анализ трёх описаний |
| Задание 5 (мультиагентная система) | 15% | Граф с тремя агентами, рабочие переходы |
| Задание 6 (RAG в мультиагентной системе) | 15% | Интеграция Chroma, поиск и использование |
| Задание 7 (сравнительный анализ) | 5% | Глубина сравнения, аргументированность |
| Задание 8 (этический протокол) | 15% | Полнота, практичность, учёт требований |
| Задание 9 (Streamlit, бонус) | +5% | Работающее приложение, демонстрация |

---

## Требования к сдаче

- Пришлите **один файл** (Jupyter Notebook `.ipynb`) со всеми заданиями, кодом и текстовыми комментариями.
- Для заданий 7 и 8 (отчёт и протокол) используйте текстовые ячейки в Notebook или приложите отдельные файлы (PDF/DOCX).
- Скриншоты вставьте в Notebook.
- Убедитесь, что код выполняется без ошибок (указаны версии библиотек, если требуется).
- Все результаты анализа должны сопровождаться интерпретацией с точки зрения врача-стоматолога (или соответствующего специалиста).

---

## Заключение

Данная практическая работа проведёт вас через полный цикл создания агентных систем для стоматологии — от установки Ollama до построения мультиагентной диагностики с настоящим RAG. Вы не только освоите технические инструменты (LangChain, LangGraph, Chroma), но и научитесь критически оценивать этические аспекты автономных ИИ-систем в клинической стоматологии.

**Главный вывод:** агенты — это мощный инструмент, расширяющий возможности врача-стоматолога и смежных специалистов, но их применение требует строгого контроля, прозрачности и ответственности. Будущее — за гибридными системами, где ИИ помогает, но человек принимает финальные клинические решения.

---

**Срок выполнения: 2 недели.**